In [11]:
import sys

!{sys.executable} -m pip install pygame

In [12]:
import pygame
print(f"Pygame version: {pygame.version.ver}")

Pygame version: 2.6.1


In [14]:
from pyvirtualdisplay import Display

display = Display(visible=0, size=(800, 600))
display.start()

Maintenant, nous allons relancer le jeu avec le display virtuel activé. Pour voir le résultat du jeu, nous devrons le capturer sous forme d'une vidéo après l'exécution. Notez que vous ne pourrez pas interagir avec le jeu en temps réel avec les touches de votre clavier dans Colab de cette manière, mais vous verrez une démonstration de son fonctionnement.

### Jeu de Plateforme Simplifié (type Mario)

Voici une version simplifiée d'un jeu de plateforme, où un carré rouge se déplace et saute. Il y aura de la gravité et des plateformes sur lesquelles il peut atterrir. Le mouvement sera automatisé pour la démonstration vidéo.

In [15]:
from pyvirtualdisplay import Display
import pygame
import numpy as np
from IPython.display import HTML
from base64 import b64encode
import imageio

# Start virtual display if not already started
try:
    display.stop()
except NameError:
    pass # display was not defined yet
display = Display(visible=0, size=(800, 600))
display.start()

# Initialize Pygame
pygame.init()

# Screen dimensions
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
SCREEN = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Simple Platformer Game")

# Colors
WHITE = (255, 255, 255)
RED = (255, 0, 0)
BLACK = (0, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)

# Game constants
GRAVITY = 0.5
JUMP_STRENGTH = -10
PLAYER_SPEED = 3

# Player properties
player_size = 40
player_x = 50
player_y = SCREEN_HEIGHT - player_size - 50 # Start above a platform
player_vel_y = 0
is_jumping = False

player_rect = pygame.Rect(player_x, player_y, player_size, player_size)

# Platforms (x, y, width, height)
platforms = [
    pygame.Rect(0, SCREEN_HEIGHT - 40, SCREEN_WIDTH, 40),  # Ground platform
    pygame.Rect(200, SCREEN_HEIGHT - 120, 150, 20),      # Floating platform 1
    pygame.Rect(450, SCREEN_HEIGHT - 200, 100, 20),      # Floating platform 2
    pygame.Rect(700, SCREEN_HEIGHT - 150, 120, 20)       # Floating platform 3
]

# Game loop
running = True
clock = pygame.time.Clock()
FPS = 60
frames = []

# Simplified movement for demonstration (automated)
move_duration = 10 * FPS # 10 seconds of movement
current_frame = 0

# Camera offset
camera_offset_x = 0

while running and current_frame < move_duration:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # --- Player movement and physics (automated) ---

    # Apply gravity
    player_vel_y += GRAVITY
    player_rect.y += player_vel_y

    # Horizontal movement (automated back and forth)
    if current_frame < move_duration / 2:
        player_rect.x += PLAYER_SPEED
    else:
        player_rect.x -= PLAYER_SPEED

    # Keep player within level bounds horizontally (basic)
    if player_rect.left < 0:
        player_rect.left = 0
    if player_rect.right > SCREEN_WIDTH * 2: # Imagine a wider level
        player_rect.right = SCREEN_WIDTH * 2

    # Collision with platforms
    on_ground = False
    for platform in platforms:
        if player_rect.colliderect(platform):
            # If falling and hit top of platform
            if player_vel_y > 0 and player_rect.bottom - player_vel_y <= platform.top:
                player_rect.bottom = platform.top
                player_vel_y = 0
                on_ground = True
                is_jumping = False # Reset jump state
            # If jumping and hit bottom of platform (not implemented for simplicity)
            elif player_vel_y < 0 and player_rect.top >= platform.bottom - player_vel_y:
                player_rect.top = platform.bottom
                player_vel_y = 0

    # Automated jump (every few seconds if on ground)
    if on_ground and np.random.randint(0, FPS * 2) == 0: # Roughly every 2 seconds
        player_vel_y = JUMP_STRENGTH
        is_jumping = True

    # Update camera offset to follow player horizontally
    camera_offset_x = max(0, player_rect.centerx - SCREEN_WIDTH // 2)
    camera_offset_x = min(camera_offset_x, (SCREEN_WIDTH * 2) - SCREEN_WIDTH) # Limit scroll

    # Drawing
    SCREEN.fill(BLUE) # Sky background

    # Draw platforms (offset by camera)
    for platform in platforms:
        draw_rect = platform.move(-camera_offset_x, 0)
        pygame.draw.rect(SCREEN, GREEN, draw_rect)

    # Draw player (offset by camera)
    draw_player_rect = player_rect.move(-camera_offset_x, 0)
    pygame.draw.rect(SCREEN, RED, draw_player_rect)

    # Capture frame
    frame = pygame.surfarray.array3d(SCREEN)
    frames.append(frame)

    # Update display
    pygame.display.flip()

    # Cap the frame rate
    clock.tick(FPS)
    current_frame += 1

# Quit Pygame
pygame.quit()
display.stop()

# Convert frames to video and display
imageio.mimsave('platformer_game.mp4', frames, fps=FPS)

mp4 = open('platformer_game.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"<video width=800 controls><source src='{data_url}' type='video/mp4'></video>")